# Module 05: Interactive Decorators, Generators & Context Managers

Welcome to **Module 05**! This laboratory dives into Python's three most powerful functional abstractions:
1. **First-Class Functions & Closures:** Inspecting `__closure__` and variable capture.
2. **Decorator Mechanics:** Wrapping functions, preserving metadata with `@functools.wraps`.
3. **Decorator Factories:** Decorators that accept arguments and configuration.
4. **Generator Pipelines & Lazy Evaluation:** Measuring massive memory savings ($O(1)$ vs $O(N)$).
5. **Context Managers:** Resource safety with `__enter__` / `__exit__` and `@contextlib.contextmanager`.
6. **Hands-On Challenge:** Implementing an active rate-limiting and execution-time decorator.


## 1. Closures & Scope Inspection


In [ ]:
def make_multiplier(factor: int):
    """Factory function creating a closure."""
    def multiply(number: int) -> int:
        return number * factor  # 'factor' is captured from the enclosing scope
    return multiply

times3 = make_multiplier(3)
times5 = make_multiplier(5)

print("times3(10):", times3(10))
print("times5(10):", times5(10))

# Inspecting the captured closure cell:
print("Captured cell contents:", times3.__closure__[0].cell_contents)


## 2. Function Decorators with Metadata Preservation


In [ ]:
import functools
import time


def timing_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        duration_ms = (time.perf_counter() - t0) * 1000
        print(f"[METRICS] {func.__name__} executed in {duration_ms:.3f} ms")
        return result
    return wrapper

@timing_decorator
def calculate_squares(limit: int) -> int:
    """Calculate sum of squares up to limit."""
    return sum(i * i for i in range(limit))

total = calculate_squares(100_000)
print("Result:", total)
# Metadata is preserved thanks to @functools.wraps:
print("Function name:", calculate_squares.__name__)
print("Docstring:", calculate_squares.__doc__)


## 3. Parameterized Decorator Factories


In [ ]:
def retry(max_attempts: int = 3, delay_sec: float = 0.05):
    """Decorator factory that retries an operation on failure."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < max_attempts:
                try:
                    return func(*args, **kwargs)
                except Exception as err:
                    attempts += 1
                    print(f"[RETRY] Attempt {attempts}/{max_attempts} failed: {err}")
                    if attempts >= max_attempts:
                        raise
                    time.sleep(delay_sec)
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=3, delay_sec=0.01)
def transient_network_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionResetError("Temporary network blip")
    return "SUCCESS_DATA"

data = transient_network_call()
print("Final result:", data)


## 4. Generator Pipelines & Memory Benchmarks


In [ ]:
import sys

# Comparing memory footprint of a List vs a Generator for 1,000,000 items:
list_data = [x * 2 for x in range(1_000_000)]
gen_data = (x * 2 for x in range(1_000_000))

list_mem = sys.getsizeof(list_data)
gen_mem = sys.getsizeof(gen_data)

print(f"List Memory:      {list_mem:,} bytes (~{list_mem / (1024*1024):.2f} MB)")
print(f"Generator Memory: {gen_mem:,} bytes")
print(f"Memory Reduction: {((list_mem - gen_mem) / list_mem) * 100:.2f}% savings!")

# Generator pipeline: filter -> transform -> aggregate
def filter_evens(stream):
    for val in stream:
        if val % 2 == 0:
            yield val

def square_values(stream):
    for val in stream:
        yield val * val

pipeline = square_values(filter_evens(range(10)))
print("Pipeline output:", list(pipeline))


## 5. Context Managers: Resource Safety


In [ ]:
from contextlib import contextmanager


# Approach A: Class-based context manager (__enter__ / __exit__)
class DatabaseSession:
    def __init__(self, db_name: str):
        self.db_name = db_name

    def __enter__(self):
        print(f"[CONNECT] Opened session to {self.db_name}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"[ROLLBACK] Rolling back transaction due to: {exc_val}")
        else:
            print(f"[COMMIT] Committing transaction on {self.db_name}")
        print("[DISCONNECT] Session closed.")
        return False  # Propagate exception if any

with DatabaseSession("analytics_db"):
    print("  Executing analytical query...")

# Approach B: Generator-based context manager (@contextmanager)
@contextmanager
def temporary_flag(store: dict, key: str, temp_value):
    old_value = store.get(key)
    store[key] = temp_value
    try:
        yield
    finally:
        if old_value is None:
            store.pop(key, None)
        else:
            store[key] = old_value

config = {"FEATURE_FLAG_V2": False}
print("Before context:", config)
with temporary_flag(config, "FEATURE_FLAG_V2", True):
    print("Inside context:", config)
print("After context:", config)


## 6. Interactive Challenge: Execution Limiter Decorator


In [ ]:
# CHALLENGE: Build a call_budget decorator that only allows a function
# to be called a maximum of N times. On the (N+1)th call, it must raise a RuntimeError!

def call_budget(max_calls: int):
    def decorator(func):
        calls_made = 0
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal calls_made
            if calls_made >= max_calls:
                raise RuntimeError(f"Execution budget exhausted! Limit is {max_calls}")
            calls_made += 1
            return func(*args, **kwargs)
        return wrapper
    return decorator

@call_budget(max_calls=2)
def access_secure_vault():
    return "VAULT_OPEN"

assert access_secure_vault() == "VAULT_OPEN"
assert access_secure_vault() == "VAULT_OPEN"

try:
    access_secure_vault()
    print("FAILED: Budget was not enforced!")
except RuntimeError as err:
    print(f"[SUCCESS] Budget enforced correctly: {err}")
